# Durable execution

A technique in which a process or workflow saves its progress at key points, allowing it to pause and later resume exactly where it left off.

By preserving completed work, durable execution enables a process to resume without reprocessing previous steps — even after a significant delay.

### Requirements

To leverage durable execution in LangGraph, we **MUST** to:
- Enable **persistence** in workflow by specifying a **checkpointer**.
- Always have a **thread identifier**
- Wrap any non-deterministic operations (e.g., random number generation) or operations with side effects (e.g., file writes, API calls) inside **tasks** to ensure that when a workflow is resumed, these operations are not repeated for the particular run, and instead their results are retrieved from the persistence layer.

### Determinism and consistent replay

When you resume a workflow run, the code does NOT resume from the same line of code where execution stopped; instead, it will identify an appropriate starting point from which to pick up where it left off.

That why we need to use **tasks** or **node** to wrap any non-deterministic operation. To ensure that your workflow is deterministic and can be consistently replayed, follow these guidelines:
- **Avoid Repeating Work**
- **Encapsulate Non-Deterministic Operation**
- **Use Idempotent Operations**

### Durability modes

LangGraph supports three durability modes that allow you to balance performance and data consistency based on your application’s requirements. A higher durability mode adds more overhead to the workflow execution.

```python
graph.stream(
    {"input": "test"},
    durability="sync"
)

From least to most durable, are as follows:
- `"exit"`: LangGraph persists changes only when graph execution exits either successfully, with an error, or due to a human in the loop interrupt.
- `"async"`: LangGraph persists changes asynchronously while the next step executes.
- `"sync"`: LangGraph persists changes synchronously before the next step starts.